In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('datos_conjuntos.csv', low_memory=False)

In [4]:
# Crea un id para cada partido
df['MATCH_ID'] = df.index
df = df.rename(columns={'1STON_FAVOURITE': '1STWON_FAVOURITE', '2NDON_FAVOURITE': '2NDWON_FAVOURITE'})

# Aces por punto jugado al servicio
df['ACE_PER_SERVE_FAVOURITE'] = np.where(df['SVPT_FAVOURITE'] > 0, df['ACE_FAVOURITE'] / df['SVPT_FAVOURITE'], 0)
df['ACE_PER_SERVE_UNDERDOG'] = np.where(df['SVPT_UNDERDOG'] > 0, df['ACE_UNDERDOG'] / df['SVPT_UNDERDOG'], 0)

# Dobles faltas por punto jugado al servicio
df['DF_PER_SERVE_FAVOURITE'] = np.where(df['SVPT_FAVOURITE'] > 0, df['DF_FAVOURITE'] / df['SVPT_FAVOURITE'], 0)
df['DF_PER_SERVE_UNDERDOG'] = np.where(df['SVPT_UNDERDOG'] > 0, df['DF_UNDERDOG'] / df['SVPT_UNDERDOG'], 0)

# % Puntos de break salvados
df['BPSAVED_PERCENTAGE_FAVOURITE'] = np.where(df['BPFACED_FAVOURITE'] > 0, df['BPSAVED_FAVOURITE'] / df['BPFACED_FAVOURITE'], 0)
df['BPSAVED_PERCENTAGE_UNDERDOG'] = np.where(df['BPFACED_UNDERDOG'] > 0, df['BPSAVED_UNDERDOG'] / df['BPFACED_UNDERDOG'], 0)

# Puntos de break generados en contra por juego jugado al servicio
df['BPFACED_PER_SVGM_FAVOURITE'] = np.where(df['SVGMS_FAVOURITE'] > 0, df['BPFACED_FAVOURITE'] / df['SVGMS_FAVOURITE'], 0)
df['BPFACED_PER_SVGM_UNDERDOG'] = np.where(df['SVGMS_UNDERDOG'] > 0, df['BPFACED_UNDERDOG'] / df['SVGMS_UNDERDOG'], 0)

# Puntos de break convertidos
df['BP_CONVERTED_FAVOURITE'] = df['BPFACED_UNDERDOG'] - df['BPSAVED_UNDERDOG']
df['BP_CONVERTED_UNDERDOG'] = df['BPFACED_FAVOURITE'] - df['BPSAVED_FAVOURITE']

# % Puntos de break convertidos
df['BP_CONVERTED_PCT_FAVOURITE'] = np.where(df['BPFACED_UNDERDOG'] > 0, df['BP_CONVERTED_FAVOURITE'] / df['BPFACED_UNDERDOG'], 0)
df['BP_CONVERTED_PCT_UNDERDOG'] = np.where(df['BPFACED_FAVOURITE'] > 0, df['BP_CONVERTED_UNDERDOG'] / df['BPFACED_FAVOURITE'], 0)

# % de Saques rotos por juego (Breaks per Return Game)
df['BREAKS_PER_SVGM_FAVOURITE'] = np.where(df['SVGMS_UNDERDOG'] > 0, df['BP_CONVERTED_FAVOURITE'] / df['SVGMS_UNDERDOG'], 0)
df['BREAKS_PER_SVGM_UNDERDOG'] = np.where(df['SVGMS_FAVOURITE'] > 0, df['BP_CONVERTED_UNDERDOG'] / df['SVGMS_FAVOURITE'], 0)

# Puntos ganados al resto
df['RETURN_PTS_WON_FAVOURITE'] = df['SVPT_UNDERDOG'] - (df['1STWON_UNDERDOG'] + df['2NDWON_UNDERDOG'])
df['RETURN_PTS_WON_UNDERDOG'] = df['SVPT_FAVOURITE'] - (df['1STWON_FAVOURITE'] + df['2NDWON_FAVOURITE'])

# % Puntos ganados al resto
df['RETURN_PTS_WON_PCT_FAVOURITE'] = np.where(df['SVPT_UNDERDOG'] > 0, df['RETURN_PTS_WON_FAVOURITE'] / df['SVPT_UNDERDOG'], 0)
df['RETURN_PTS_WON_PCT_UNDERDOG'] = np.where(df['SVPT_FAVOURITE'] > 0, df['RETURN_PTS_WON_UNDERDOG'] / df['SVPT_FAVOURITE'], 0)

# Total de segundos saques
df['2ND_SERVES_FAVOURITE'] = df['SVPT_FAVOURITE'] - df['1STIN_FAVOURITE']
df['2ND_SERVES_UNDERDOG'] = df['SVPT_UNDERDOG'] - df['1STIN_UNDERDOG']

# Puntos ganados sobre el segundo saque del rival
df['RETURN_2ND_WON_FAVOURITE'] = df['2ND_SERVES_UNDERDOG'] - df['2NDWON_UNDERDOG']
df['RETURN_2ND_WON_UNDERDOG'] = df['2ND_SERVES_FAVOURITE'] - df['2NDWON_FAVOURITE']

# % Puntos ganados sobre el segundo saque del rival
df['RETURN_2ND_WON_PCT_FAVOURITE'] = np.where(df['2ND_SERVES_UNDERDOG'] > 0, df['RETURN_2ND_WON_FAVOURITE'] / df['2ND_SERVES_UNDERDOG'], 0)
df['RETURN_2ND_WON_PCT_UNDERDOG'] = np.where(df['2ND_SERVES_FAVOURITE'] > 0, df['RETURN_2ND_WON_UNDERDOG'] / df['2ND_SERVES_FAVOURITE'], 0)

games_fav = ['1_FAVOURITE', '2_FAVOURITE', '3_FAVOURITE', '4_FAVOURITE', '5_FAVOURITE']
games_und = ['1_UNDERDOG', '2_UNDERDOG', '3_UNDERDOG', '4_UNDERDOG', '5_UNDERDOG']

# Juegos ganados
df['TOTAL_GAMES_WON_FAVOURITE'] = df[games_fav].fillna(0).sum(axis=1)
df['TOTAL_GAMES_WON_UNDERDOG'] = df[games_und].fillna(0).sum(axis=1)

df['TOTAL_GAMES'] = df['TOTAL_GAMES_WON_FAVOURITE'] + df['TOTAL_GAMES_WON_UNDERDOG']

# % Juegos ganados
df['GAMES_WON_PCT_FAVOURITE'] = np.where(df['TOTAL_GAMES'] > 0, df['TOTAL_GAMES_WON_FAVOURITE'] / df['TOTAL_GAMES'], 0)
df['GAMES_WON_PCT_UNDERDOG'] = np.where(df['TOTAL_GAMES'] > 0, df['TOTAL_GAMES_WON_UNDERDOG'] / df['TOTAL_GAMES'], 0)

In [5]:
# Numero de partidos a considerar para el análisis de estadísticas recientes
last_matches = [1,3,5,10]

# Variables de interés para el análisis
stats = ['ACE_PER_SERVE','DF_PER_SERVE','BPSAVED_PERCENTAGE','BPFACED_PER_SVGM','BP_CONVERTED_PCT','BREAKS_PER_SVGM','RETURN_PTS_WON_PCT',
         'RETURN_2ND_WON_PCT','GAMES_WON_PCT','1STIN','1STWON','2NDWON']

In [6]:
base_cols = ['MATCH_ID', 'SURFACE', 'COURT', 'MINUTES', 'UPSET']

# Columnas del jugador favorito
favourite_cols = base_cols + ['ID_FAVOURITE'] + [f"{s}_FAVOURITE" for s in stats]
df_fav = df[favourite_cols].copy()
df_fav.columns = base_cols + ['PLAYER_ID'] + stats
df_fav['WON'] = (df_fav['UPSET'] == 0).astype(int)

# Columnas del jugador no favorito
underdog_cols = base_cols + ['ID_UNDERDOG'] + [f"{s}_UNDERDOG" for s in stats]
df_und = df[underdog_cols].copy()
df_und.columns = base_cols + ['PLAYER_ID'] + stats
df_und['WON'] = (df_und['UPSET'] == 1).astype(int)

# Dupicamos las filas para que cada una represente a un jugador en un partido.
# Permite calcular datos sin tener que saber si el jugador era favorito o underdog
df_long = pd.concat([df_fav, df_und]).sort_values(by=['MATCH_ID'])
stats_to_roll = stats + ['WON']
df_long.head()

,MATCH_ID,SURFACE,COURT,MINUTES,UPSET,PLAYER_ID,ACE_PER_SERVE,DF_PER_SERVE,BPSAVED_PERCENTAGE,BPFACED_PER_SVGM,BP_CONVERTED_PCT,BREAKS_PER_SVGM,RETURN_PTS_WON_PCT,RETURN_2ND_WON_PCT,GAMES_WON_PCT,1STIN,1STWON,2NDWON,WON
0,0,Hard,NaN,130.0,1,101414,0.084211,0.031579,0.75,0.500000,0.500,0.133333,0.333333,0.390244,0.50,62.0,44.0,23.0,0
0,0,Hard,NaN,130.0,1,101723,0.062500,0.020833,0.50,0.266667,0.250,0.125000,0.294737,0.303030,0.50,55.0,39.0,25.0,1
1,1,Hard,NaN,71.0,1,101421,0.033333,0.033333,0.50,1.000000,0.000,0.000000,0.314815,0.434783,0.25,37.0,22.0,6.0,0
1,1,Hard,NaN,71.0,1,101234,0.111111,0.018519,1.00,0.125000,0.500,0.500000,0.533333,0.739130,0.75,31.0,24.0,13.0,1
2,2,Hard,NaN,85.0,0,101889,0.033333,0.000000,1.00,0.333333,0.375,0.300000,0.445946,0.620690,0.65,40.0,30.0,14.0,1


In [7]:
# Tiempo de ejecución: 5 minutos aprox.

# Cálculo del h2h

# Estandarizamos los ids de los jugadores para que P1 siempre sea el jugador con menor id y P2 el jugador con mayor id,
# sin importar si es favorito o underdog.
df['ID_P1'] = np.minimum(df['ID_FAVOURITE'], df['ID_UNDERDOG'])
df['ID_P2'] = np.maximum(df['ID_FAVOURITE'], df['ID_UNDERDOG'])

p1_fav = (df['ID_P1'] == df['ID_FAVOURITE'])
fav_won = (df['UPSET'] == 0)

df['P1_WON'] = np.where((p1_fav & fav_won) | (~p1_fav & ~fav_won), 1, 0)

new_cols_h2h = {}

for n in last_matches:
    # h2h general
    new_cols_h2h[f'P1_WON_H2H_LAST_{n}'] = df.groupby(['ID_P1', 'ID_P2'])['P1_WON'].transform(
        lambda x: x.shift(1).rolling(window=n, min_periods=1).mean()
    )
    # h2h por superficie
    new_cols_h2h[f'P1_WON_H2H_SURFACE_{n}'] = df.groupby(['ID_P1', 'ID_P2', 'SURFACE'])['P1_WON'].transform(
        lambda x: x.shift(1).rolling(window=n, min_periods=1).mean()
    )
    # h2h por superficie y pista
    new_cols_h2h[f'P1_WON_H2H_COURT_{n}'] = df.groupby(['ID_P1', 'ID_P2', 'SURFACE', 'COURT'])['P1_WON'].transform(
        lambda x: x.shift(1).rolling(window=n, min_periods=1).mean()
    )

df = pd.concat([df, pd.DataFrame(new_cols_h2h)], axis=1)

# Traduce el % de victorias de P1 a Favorito (Underdog no hace falta ya que simplemente es la inversa)
for n in last_matches:
    for suffix in [f'LAST_{n}', f'SURFACE_{n}', f'COURT_{n}']:
        p1_col = f'P1_WON_H2H_{suffix}'
        fav_col = f'FAV_H2H_WON_{suffix}'
        
        df[fav_col] = df[p1_col]
        p2_fav = (df['ID_FAVOURITE'] == df['ID_P2'])
        df.loc[p2_fav, fav_col] = 1 - df.loc[p2_fav, p1_col]

In [8]:
# Borramos las columnas temporales del h2h
cols_to_drop = ['ID_P1', 'ID_P2', 'P1_WON'] + list(new_cols_h2h.keys())
df = df.drop(columns=cols_to_drop)

In [9]:
# Tiempo de ejecución: 4 minutos aprox.

cols_generated = []
new_cols = {}

# Calculamos los promedios de las estadísticas de interés para los últimos n partidos, por jugador, superficie y tipo de pista.

for n in last_matches:
    # Minutos en pista del jugador
    col_minutes = f'MINUTES_LAST_{n}'
    new_cols[col_minutes] = df_long.groupby('PLAYER_ID')['MINUTES'].transform(
        lambda x: x.shift(1).rolling(window=n, min_periods=1).mean()
    )
    cols_generated.append(col_minutes)

    for stat in stats_to_roll:
        # A. Promedio general de los últimos n partidos
        col_general = f'{stat}_LAST_{n}'
        new_cols[col_general] = df_long.groupby('PLAYER_ID')[stat].transform(
            lambda x: x.shift(1).rolling(window=n, min_periods=1).mean()
        )
        cols_generated.append(col_general)
        
        # B. Promedio por superficie de los últimos n partidos
        col_surface = f'{stat}_SURFACE_{n}'
        new_cols[col_surface] = df_long.groupby(['PLAYER_ID', 'SURFACE'])[stat].transform(
            lambda x: x.shift(1).rolling(window=n, min_periods=1).mean()
        )
        cols_generated.append(col_surface)
        
        # C. Promedio por superficie y tipo de pista de los últimos n partidos
        col_court = f'{stat}_COURT_{n}'
        new_cols[col_court] = df_long.groupby(['PLAYER_ID', 'SURFACE', 'COURT'])[stat].transform(
            lambda x: x.shift(1).rolling(window=n, min_periods=1).mean()
        )
        cols_generated.append(col_court)

df_new_cols = pd.DataFrame(new_cols)
df_long = pd.concat([df_long, df_new_cols], axis=1)

# Separamos y volvemos a unir al df original
cols_merge = ['MATCH_ID', 'PLAYER_ID'] + cols_generated

# Para el favorito
df_fav_merge = df_long[cols_merge].copy()
df_fav_merge.columns = ['MATCH_ID', 'ID_FAVOURITE'] + [f'FAV_{c}' for c in cols_generated]

# Para el underdog
df_und_merge = df_long[cols_merge].copy()
df_und_merge.columns = ['MATCH_ID', 'ID_UNDERDOG'] + [f'UND_{c}' for c in cols_generated]

# 5. Unimos al dataframe principal
df = df.merge(df_fav_merge, on=['MATCH_ID', 'ID_FAVOURITE'], how='left')
df = df.merge(df_und_merge, on=['MATCH_ID', 'ID_UNDERDOG'], how='left')

In [10]:
# Calcula diferencias o ratios entre jugadores de posible interés

# Diferencia absoluta de altura (variable lineal)
df['DIFF_HT'] = df['HT_FAVOURITE'] - df['HT_UNDERDOG']

# Diferencia absoluta de edad
df['DIFF_AGE'] = df['AGE_FAVOURITE'] - df['AGE_UNDERDOG']

# Ratio de diferencia de puntos ATP (+1 para evitar dividir por cero y evitar valores extremos)
df['RATIO_POINTS'] = df['RANK_POINTS_FAVOURITE'] / (df['RANK_POINTS_UNDERDOG'] + 1)

# Ratio de diferencia de ranking ATP (+1 para evitar dividir por cero y evitar valores extremos)
df['RATIO_RANK'] = df['RANK_UNDERDOG'] / (df['RANK_FAVOURITE'] + 1)

# ¿Hay diferencia de mano dominante?
df['FAV_R_UND_L'] = ((df['HAND_FAVOURITE'] == 'R') & (df['HAND_UNDERDOG'] == 'L')).astype(int)
df['FAV_L_UND_R'] = ((df['HAND_FAVOURITE'] == 'L') & (df['HAND_UNDERDOG'] == 'R')).astype(int)

In [11]:
df.head()

,TOURNEY_ID,TOURNEY_NAME,SURFACE,DRAW_SIZE,TOURNEY_DATE,SCORE,BEST_OF,ROUND,MINUTES,SERIES,...,UND_2NDWON_COURT_10,UND_WON_LAST_10,UND_WON_SURFACE_10,UND_WON_COURT_10,DIFF_HT,DIFF_AGE,RATIO_POINTS,RATIO_RANK,FAV_R_UND_L,FAV_L_UND_R
0,1991-339,Adelaide,Hard,32,1990-12-31,6-4 3-6 7-6(2),3,R32,130.0,NaN,...,NaN,NaN,NaN,NaN,-3.0,2.4,NaN,18.666667,0,0
1,1991-339,Adelaide,Hard,32,1990-12-31,6-0 6-4,3,R32,71.0,NaN,...,NaN,NaN,NaN,NaN,-7.0,-1.8,NaN,1.171429,0,0
2,1991-339,Adelaide,Hard,32,1990-12-31,7-6(2) 6-1,3,R32,85.0,NaN,...,NaN,NaN,NaN,NaN,-2.0,-1.1,NaN,1.647059,0,0
3,1991-339,Adelaide,Hard,32,1990-12-31,7-5 6-3,3,R32,90.0,NaN,...,NaN,NaN,NaN,NaN,3.0,-4.5,NaN,3.034483,0,0
4,1991-339,Adelaide,Hard,32,1990-12-31,6-2 6-3,3,R32,88.0,NaN,...,NaN,NaN,NaN,NaN,5.0,6.3,NaN,1.033333,0,0


In [13]:
df['TOURNEY_DATE'] = pd.to_datetime(df['TOURNEY_DATE'])
df['YEAR'] = df['TOURNEY_DATE'].dt.year

In [14]:
from sklearn.preprocessing import LabelEncoder

# Codificamos las variables categóricas de interés

rounds_order = ['RR', 'BR', 'ER', 'R128', 'R64', 'R32', 'R16', 'QF', 'SF', 'F']
series_order = ['ATP 250', 'ATP 500', 'Masters 1000', 'ATP Finals', 'Grand Slam']

rounds_mapping = {val: i for i, val in enumerate(rounds_order)}
series_mapping = {val: i for i, val in enumerate(series_order)}

df['ROUND'] = df['ROUND'].map(rounds_mapping)
df['SERIES'] = df['SERIES'].map(series_mapping)

nominal_cols = ['HAND_FAVOURITE', 'HAND_UNDERDOG', 'IOC_FAVOURITE', 'IOC_UNDERDOG']
encoder = LabelEncoder()
for col in nominal_cols:
    if col in df.columns:
        df[col] = encoder.fit_transform(df[col].astype(str))

In [15]:
df.to_csv('datos_finales.csv', index=False, encoding='latin1')